# 银行数据库 重构

---

本Notebook用于银行财务数据的存储和管理，为银行债定价模型、信用风险评估等分析提供数据支持。

## 1. 项目概述

### 1.1 功能描述

**核心功能**：
1. **数据存储**：存储和管理各家银行的财务指标数据
2. **数据支持**：为银行债定价模型、信用风险评估等分析提供数据支持
3. **数据导入**：将Excel数据导入到数据库
4. **增量更新**：支持数据的增量更新机制

**数据字段**：
- `dt1`: 日期
- `发行人`: 银行发行人名称
- `总资产`: 银行总资产（亿元）
- `资本充足率`: 资本充足率（%）
- `净息差`: 净利息收入与平均生息资产的比率（%）
- `不良率`: 不良贷款率（%）
- `ROE`: 净资产收益率（%）
- `存款占比`: 存款占总负债的比例（%）
- `拨备覆盖率`: 贷款损失准备金与不良贷款的比率（%）

### 1.2 数据源

**数据文件**：
- `银行数据库2024.xlsx`：2024年银行财务数据主文件（约704KB）
- `银行.xlsx`：模板文件或示例数据（约26KB）
- `副本银行数据库2024 - 副本.xlsx`：完整备份副本（约3.13MB）

**数据来源**：
- 银行财报
- 监管机构披露
- 第三方数据提供商（如Wind、同花顺）

**更新频率**：季度更新

### 1.3 输出结果

1. **数据库表**：`yq.银行数据库`
2. **数据验证报告**：数据完整性和一致性检查结果
3. **更新日志**：记录每次数据更新的详情

## 2. 环境配置

### 2.1 导入依赖

In [ ]:
# 标准库
import os
import sys
import datetime
import logging
from typing import List, Dict, Optional, Tuple

# 第三方库
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from sqlalchemy.types import VARCHAR, DECIMAL, DATE, FLOAT
import pymysql
import openpyxl

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    encoding='utf-8'
)
logger = logging.getLogger(__name__)

print("依赖导入完成")

### 2.2 配置参数

In [ ]:
# 从config模块导入配置
from config import (
    DATABASE_HOST,
    DATABASE_PORT,
    DATABASE_USER,
    DATABASE_PASSWORD,
    DATABASE_NAME,
    DATA_DIR,
    OUTPUT_DIR,
    MAIN_DATA_FILE,
    BACKUP_DATA_FILE,
    TABLE_NAME,
    get_database_url
)

# 打印配置信息（隐藏敏感信息）
print(f"数据库主机: {DATABASE_HOST}")
print(f"数据库端口: {DATABASE_PORT}")
print(f"数据库名称: {DATABASE_NAME}")
print(f"数据目录: {DATA_DIR}")
print(f"输出目录: {OUTPUT_DIR}")
print(f"目标表名: {TABLE_NAME}")

## 3. 数据获取

### 3.1 读取Excel数据

In [ ]:
def load_bank_data(file_path: str, sheet_name: str = 0) -> pd.DataFrame:
    """
    读取银行财务数据Excel文件
    
    Parameters
    ----------
    file_path : str
        Excel文件路径
    sheet_name : str or int, default 0
        工作表名称或索引
    
    Returns
    -------
    pd.DataFrame
        银行财务数据
    """
    try:
        df = pd.read_excel(file_path, sheet_name=sheet_name)
        logger.info(f"成功读取文件: {file_path}")
        logger.info(f"数据形状: {df.shape}")
        return df
    except Exception as e:
        logger.error(f"读取文件失败: {file_path}, 错误: {e}")
        raise

# 读取主数据文件
main_file_path = os.path.join(DATA_DIR, MAIN_DATA_FILE)
if os.path.exists(main_file_path):
    df_bank = load_bank_data(main_file_path)
    print(f"\n数据预览:")
    display(df_bank.head())
    print(f"\n列名:")
    print(df_bank.columns.tolist())
    print(f"\n数据类型:")
    print(df_bank.dtypes)
else:
    print(f"警告: 主数据文件不存在: {main_file_path}")
    df_bank = None

### 3.2 数据库连接

In [ ]:
def create_db_connection() -> Tuple:
    """
    创建数据库连接
    
    Returns
    -------
    Tuple
        (SQLAlchemy Engine, pymysql connection)
    """
    try:
        # 创建SQLAlchemy引擎
        db_url = get_database_url()
        engine = create_engine(
            db_url,
            pool_recycle=3600,
            pool_pre_ping=True,
            echo=False
        )
        
        # 测试连接
        with engine.connect() as conn:
            conn.execute(text("SELECT 1"))
        
        logger.info("数据库连接成功")
        return engine
    except Exception as e:
        logger.error(f"数据库连接失败: {e}")
        raise

# 创建数据库连接
try:
    db_engine = create_db_connection()
    print("数据库连接成功")
except Exception as e:
    print(f"数据库连接失败: {e}")
    db_engine = None

## 4. 数据处理

### 4.1 数据验证

In [ ]:
def validate_bank_data(df: pd.DataFrame) -> Dict:
    """
    验证银行财务数据的完整性和合理性
    
    Parameters
    ----------
    df : pd.DataFrame
        银行财务数据
    
    Returns
    -------
    Dict
        验证结果报告
    """
    report = {
        'total_rows': len(df),
        'total_columns': len(df.columns),
        'missing_values': {},
        'duplicate_rows': 0,
        'validation_errors': [],
        'warnings': []
    }
    
    # 检查必需字段
    required_columns = ['dt1', '发行人']
    for col in required_columns:
        if col not in df.columns:
            report['validation_errors'].append(f"缺少必需字段: {col}")
    
    # 检查缺失值
    for col in df.columns:
        missing_count = df[col].isna().sum()
        if missing_count > 0:
            report['missing_values'][col] = int(missing_count)
    
    # 检查重复行
    if 'dt1' in df.columns and '发行人' in df.columns:
        report['duplicate_rows'] = df.duplicated(subset=['dt1', '发行人']).sum()
    
    # 检查数值范围合理性
    range_checks = [
        ('资本充足率', 0, 50),
        ('不良率', 0, 20),
        ('ROE', -50, 50),
        ('净息差', -5, 10),
        ('存款占比', 0, 100),
        ('拨备覆盖率', 0, 1000),
    ]
    
    for col, min_val, max_val in range_checks:
        if col in df.columns:
            out_of_range = ((df[col] < min_val) | (df[col] > max_val)).sum()
            if out_of_range > 0:
                report['warnings'].append(
                    f"{col}: 有{out_of_range}条数据超出合理范围[{min_val}, {max_val}]"
                )
    
    return report

# 验证数据
if df_bank is not None:
    validation_report = validate_bank_data(df_bank)
    print("数据验证报告:")
    print(f"  总行数: {validation_report['total_rows']}")
    print(f"  总列数: {validation_report['total_columns']}")
    print(f"  重复行数: {validation_report['duplicate_rows']}")
    print(f"  缺失值: {validation_report['missing_values']}")
    if validation_report['validation_errors']:
        print(f"  验证错误: {validation_report['validation_errors']}")
    if validation_report['warnings']:
        print(f"  警告: {validation_report['warnings']}")

### 4.2 数据清洗

In [ ]:
def clean_bank_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    清洗银行财务数据
    
    Parameters
    ----------
    df : pd.DataFrame
        原始银行财务数据
    
    Returns
    -------
    pd.DataFrame
        清洗后的数据
    """
    df_clean = df.copy()
    
    # 标准化日期列
    if 'dt1' in df_clean.columns:
        df_clean['dt1'] = pd.to_datetime(df_clean['dt1'], errors='coerce')
    
    # 清理字符串列
    if '发行人' in df_clean.columns:
        df_clean['发行人'] = df_clean['发行人'].astype(str).str.strip()
    
    # 数值列处理
    numeric_columns = ['总资产', '资本充足率', '净息差', '不良率', 'ROE', '存款占比', '拨备覆盖率']
    for col in numeric_columns:
        if col in df_clean.columns:
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    
    # 移除日期或发行人为空的行
    if 'dt1' in df_clean.columns and '发行人' in df_clean.columns:
        df_clean = df_clean.dropna(subset=['dt1', '发行人'])
    
    # 移除重复行
    df_clean = df_clean.drop_duplicates(subset=['dt1', '发行人'], keep='last')
    
    # 按日期和发行人排序
    df_clean = df_clean.sort_values(['发行人', 'dt1']).reset_index(drop=True)
    
    logger.info(f"数据清洗完成: 原始{len(df)}行 -> 清洗后{len(df_clean)}行")
    return df_clean

# 清洗数据
if df_bank is not None:
    df_clean = clean_bank_data(df_bank)
    print(f"\n清洗后数据预览:")
    display(df_clean.head())
    print(f"\n清洗后数据统计:")
    print(df_clean.describe())

## 5. 核心逻辑

### 5.1 数据导入

In [ ]:
def import_to_database(df: pd.DataFrame, engine, table_name: str, if_exists: str = 'append') -> int:
    """
    将数据导入到数据库
    
    Parameters
    ----------
    df : pd.DataFrame
        要导入的数据
    engine : SQLAlchemy Engine
        数据库引擎
    table_name : str
        目标表名
    if_exists : str, default 'append'
        表存在时的处理方式: 'fail', 'replace', 'append'
    
    Returns
    -------
    int
        导入的行数
    """
    try:
        # 定义数据类型映射
        dtype_mapping = {
            'dt1': DATE(),
            '发行人': VARCHAR(100),
            '总资产': DECIMAL(20, 4),
            '资本充足率': DECIMAL(10, 4),
            '净息差': DECIMAL(10, 4),
            '不良率': DECIMAL(10, 4),
            'ROE': DECIMAL(10, 4),
            '存款占比': DECIMAL(10, 4),
            '拨备覆盖率': DECIMAL(10, 4),
        }
        
        # 筛选实际存在的列
        existing_columns = [col for col in dtype_mapping.keys() if col in df.columns]
        df_to_import = df[existing_columns].copy()
        dtype_to_use = {col: dtype_mapping[col] for col in existing_columns}
        
        # 导入数据
        rows_imported = df_to_import.to_sql(
            name=table_name,
            con=engine,
            if_exists=if_exists,
            index=False,
            dtype=dtype_to_use,
            chunksize=1000
        )
        
        logger.info(f"成功导入 {len(df_to_import)} 行数据到表 {table_name}")
        return len(df_to_import)
    except Exception as e:
        logger.error(f"数据导入失败: {e}")
        raise

# 导入数据（仅在数据库连接可用时执行）
if db_engine is not None and df_clean is not None:
    try:
        rows_imported = import_to_database(df_clean, db_engine, TABLE_NAME, if_exists='replace')
        print(f"成功导入 {rows_imported} 行数据")
    except Exception as e:
        print(f"导入失败: {e}")

### 5.2 增量更新

In [ ]:
def get_existing_data(engine, table_name: str) -> pd.DataFrame:
    """
    获取数据库中已存在的数据
    
    Parameters
    ----------
    engine : SQLAlchemy Engine
        数据库引擎
    table_name : str
        表名
    
    Returns
    -------
    pd.DataFrame
        已存在的数据
    """
    try:
        query = f"SELECT * FROM {table_name}"
        df_existing = pd.read_sql_query(query, engine)
        logger.info(f"获取到 {len(df_existing)} 条已存在数据")
        return df_existing
    except Exception as e:
        logger.warning(f"获取已存在数据失败（表可能不存在）: {e}")
        return pd.DataFrame()

def incremental_update(df_new: pd.DataFrame, engine, table_name: str) -> int:
    """
    增量更新数据
    
    Parameters
    ----------
    df_new : pd.DataFrame
        新数据
    engine : SQLAlchemy Engine
        数据库引擎
    table_name : str
        表名
    
    Returns
    -------
    int
        新增的行数
    """
    # 获取已存在数据
    df_existing = get_existing_data(engine, table_name)
    
    if df_existing.empty:
        # 表不存在，直接导入所有数据
        return import_to_database(df_new, engine, table_name, if_exists='replace')
    
    # 创建复合键用于比较
    if 'dt1' in df_new.columns and '发行人' in df_new.columns:
        df_new['composite_key'] = df_new['dt1'].astype(str) + '_' + df_new['发行人']
        df_existing['composite_key'] = df_existing['dt1'].astype(str) + '_' + df_existing['发行人']
        
        # 筛选出新数据
        existing_keys = set(df_existing['composite_key'])
        df_to_insert = df_new[~df_new['composite_key'].isin(existing_keys)].copy()
        df_to_insert = df_to_insert.drop(columns=['composite_key'])
        
        if len(df_to_insert) > 0:
            return import_to_database(df_to_insert, engine, table_name, if_exists='append')
        else:
            logger.info("没有新数据需要更新")
            return 0
    else:
        logger.warning("缺少dt1或发行人列，无法进行增量更新")
        return 0

# 测试增量更新（仅在数据库连接可用时执行）
if db_engine is not None and df_clean is not None:
    print("增量更新功能已准备就绪")

### 5.3 财务指标计算

In [ ]:
def calculate_financial_changes(df: pd.DataFrame, columns: List[str], periods: int = 365) -> pd.DataFrame:
    """
    计算财务指标的期间变化
    
    Parameters
    ----------
    df : pd.DataFrame
        银行财务数据
    columns : List[str]
        需要计算变化的列名
    periods : int, default 365
        变化周期（天数）
    
    Returns
    -------
    pd.DataFrame
        添加了变化列的数据
    """
    df_result = df.copy()
    
    # 确保日期列是datetime类型
    if 'dt1' in df_result.columns:
        df_result['dt1'] = pd.to_datetime(df_result['dt1'])
    
    # 按发行人分组计算变化
    for col in columns:
        if col in df_result.columns:
            change_col_name = f'近{periods//30}月{col}变化'
            df_result[change_col_name] = df_result.groupby('发行人')[col].transform(
                lambda x: x - x.shift(periods) if len(x) > periods else np.nan
            )
    
    return df_result

# 计算财务指标变化
if df_clean is not None:
    # 选择要计算变化的列
    change_columns = ['总资产', '资本充足率', '净息差', '不良率', 'ROE']
    existing_change_cols = [col for col in change_columns if col in df_clean.columns]
    
    if existing_change_cols:
        df_with_changes = calculate_financial_changes(df_clean, existing_change_cols)
        print("财务指标变化计算完成:")
        new_cols = [col for col in df_with_changes.columns if '变化' in col]
        print(f"新增变化列: {new_cols}")
        display(df_with_changes[['发行人', 'dt1'] + existing_change_cols + new_cols[:3]].head(10))

## 6. 执行与测试

### 6.1 执行主流程

In [ ]:
def main():
    """
    主执行流程
    """
    logger.info("开始执行银行数据库导入流程")
    
    try:
        # 1. 读取数据
        main_file_path = os.path.join(DATA_DIR, MAIN_DATA_FILE)
        if not os.path.exists(main_file_path):
            raise FileNotFoundError(f"数据文件不存在: {main_file_path}")
        
        df_raw = load_bank_data(main_file_path)
        logger.info(f"步骤1完成: 读取原始数据 {len(df_raw)} 行")
        
        # 2. 验证数据
        validation_report = validate_bank_data(df_raw)
        if validation_report['validation_errors']:
            raise ValueError(f"数据验证失败: {validation_report['validation_errors']}")
        logger.info("步骤2完成: 数据验证通过")
        
        # 3. 清洗数据
        df_clean = clean_bank_data(df_raw)
        logger.info(f"步骤3完成: 数据清洗完成 {len(df_clean)} 行")
        
        # 4. 导入数据库
        if db_engine is not None:
            rows_imported = import_to_database(df_clean, db_engine, TABLE_NAME, if_exists='replace')
            logger.info(f"步骤4完成: 数据导入完成 {rows_imported} 行")
        else:
            logger.warning("步骤4跳过: 数据库连接不可用")
        
        logger.info("银行数据库导入流程执行完成")
        return True
    
    except Exception as e:
        logger.error(f"执行失败: {e}")
        return False

# 执行主流程
result = main()
print(f"\n执行结果: {'成功' if result else '失败'}")

### 6.2 结果验证

In [ ]:
def verify_import(engine, table_name: str, expected_count: int = None) -> Dict:
    """
    验证数据导入结果
    
    Parameters
    ----------
    engine : SQLAlchemy Engine
        数据库引擎
    table_name : str
        表名
    expected_count : int, optional
        预期的行数
    
    Returns
    -------
    Dict
        验证结果
    """
    result = {
        'success': False,
        'row_count': 0,
        'unique_banks': 0,
        'date_range': None,
        'errors': []
    }
    
    try:
        # 检查表是否存在
        df = pd.read_sql_query(f"SELECT * FROM {table_name} LIMIT 1", engine)
        
        # 获取行数
        count_df = pd.read_sql_query(f"SELECT COUNT(*) as cnt FROM {table_name}", engine)
        result['row_count'] = int(count_df['cnt'].iloc[0])
        
        # 获取银行数量
        banks_df = pd.read_sql_query(
            f"SELECT COUNT(DISTINCT 发行人) as bank_cnt FROM {table_name}",
            engine
        )
        result['unique_banks'] = int(banks_df['bank_cnt'].iloc[0])
        
        # 获取日期范围
        date_df = pd.read_sql_query(
            f"SELECT MIN(dt1) as min_date, MAX(dt1) as max_date FROM {table_name}",
            engine
        )
        result['date_range'] = (
            str(date_df['min_date'].iloc[0]),
            str(date_df['max_date'].iloc[0])
        )
        
        # 检查预期行数
        if expected_count is not None:
            if result['row_count'] != expected_count:
                result['errors'].append(
                    f"行数不匹配: 预期{expected_count}, 实际{result['row_count']}"
                )
        
        result['success'] = len(result['errors']) == 0
        
    except Exception as e:
        result['errors'].append(str(e))
    
    return result

# 验证导入结果（仅在数据库连接可用时执行）
if db_engine is not None:
    verification = verify_import(db_engine, TABLE_NAME)
    print("导入验证结果:")
    print(f"  成功: {verification['success']}")
    print(f"  行数: {verification['row_count']}")
    print(f"  银行数量: {verification['unique_banks']}")
    print(f"  日期范围: {verification['date_range']}")
    if verification['errors']:
        print(f"  错误: {verification['errors']}")
else:
    print("数据库连接不可用，跳过验证")

## 7. 工具函数（可复用）

In [ ]:
def query_bank_data(
    engine,
    table_name: str,
    columns: List[str] = None,
    issuers: List[str] = None,
    start_date: str = None,
    end_date: str = None
) -> pd.DataFrame:
    """
    查询银行财务数据
    
    Parameters
    ----------
    engine : SQLAlchemy Engine
        数据库引擎
    table_name : str
        表名
    columns : List[str], optional
        要查询的列，默认查询所有列
    issuers : List[str], optional
        发行人列表，默认查询所有
    start_date : str, optional
        开始日期
    end_date : str, optional
        结束日期
    
    Returns
    -------
    pd.DataFrame
        查询结果
    """
    # 构建SQL查询
    if columns:
        select_clause = ', '.join([f'`{col}`' for col in columns])
    else:
        select_clause = '*'
    
    query = f"SELECT {select_clause} FROM {table_name}"
    conditions = []
    
    if issuers:
        issuer_list = ', '.join([f"'{iss}'" for iss in issuers])
        conditions.append(f"发行人 IN ({issuer_list})")
    
    if start_date:
        conditions.append(f"dt1 >= '{start_date}'")
    
    if end_date:
        conditions.append(f"dt1 <= '{end_date}'")
    
    if conditions:
        query += " WHERE " + " AND ".join(conditions)
    
    query += " ORDER BY 发行人, dt1"
    
    return pd.read_sql_query(query, engine)

def export_to_excel(df: pd.DataFrame, output_path: str, sheet_name: str = 'Sheet1') -> str:
    """
    将数据导出到Excel文件
    
    Parameters
    ----------
    df : pd.DataFrame
        要导出的数据
    output_path : str
        输出文件路径
    sheet_name : str, default 'Sheet1'
        工作表名称
    
    Returns
    -------
    str
        输出文件路径
    """
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)
    
    logger.info(f"数据已导出到: {output_path}")
    return output_path

def get_bank_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    获取银行数据汇总统计
    
    Parameters
    ----------
    df : pd.DataFrame
        银行财务数据
    
    Returns
    -------
    pd.DataFrame
        汇总统计
    """
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    
    # 排除标识列
    exclude_cols = ['dt1', '发行人']
    numeric_cols = [col for col in numeric_cols if col not in exclude_cols]
    
    summary = df[numeric_cols].describe().T
    summary['missing'] = df[numeric_cols].isna().sum()
    
    return summary

# 测试工具函数
if df_clean is not None:
    print("银行数据汇总统计:")
    summary = get_bank_summary(df_clean)
    display(summary)

---

**重构完成时间**: 2026-02-15

**任务编号**: T49

**任务名称**: 银行数据库